# Is the code computing the right matrix?

Three words of background. A **cubic spline** is a curve made of
cubic pieces glued together smoothly; the gluing points are called
**knots**, and we write them $t_0, t_1, \dots$. Every such curve is a
weighted sum of a few standard bump-shaped curves called **B-splines**,
written $B_i$: choose the weights, get the curve.

A *smoothing* spline picks the curve by penalizing wiggliness, and the
penalty is a matrix $\Omega$. It can be written two ways.

What it is: $\Omega_{ij} = \int B_i''\, B_j''$. Take two of the
bumps, differentiate each twice (curvature), multiply, integrate.

How to build it: $\Omega = C^\top R\, C$, three small matrices made
directly from the knots, no integration. This is what the code below
does, and a
[20-page report](https://github.com/aadya940/scipy-bspline-testing/blob/main/B_Splines_with_arbitary_knots-gcv.pdf)
derives why the two are the same thing.

Let's check that the code and the integral agree, without trusting the
report.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, sympy
from IPython.display import Math, display
from skverify import to_sympy, latex
from skverify.helpers import axis_idx

def penalty(t):
    m = len(t) - 4
    D1 = np.zeros((m + 1, m))
    for j in range(m + 1):
        run = t[j + 3] - t[j]
        if run != 0:
            if j - 1 >= 0:
                D1[j, j - 1] = -3 / run
            if j < m:
                D1[j, j] = 3 / run
    D2 = np.zeros((m + 2, m + 1))
    for j in range(m + 2):
        run = t[j + 2] - t[j]
        if run != 0:
            if j - 1 >= 0:
                D2[j, j - 1] = -2 / run
            if j < m + 1:
                D2[j, j] = 2 / run
    C = D2 @ D1
    R = np.zeros((m + 2, m + 2))
    for p in range(m + 2):
        R[p, p] = (t[p + 2] - t[p]) / 3
        if p + 1 < m + 2:
            R[p, p + 1] = R[p + 1, p] = (t[p + 2] - t[p + 1]) / 6
    return C.T @ R @ C

Run it once through skverify. The knots become symbols
$t_0, \dots, t_8$, and every entry of the result becomes a formula in
them. Here is entry $(1,2)$ of what the code computed:

In [2]:
t9 = np.array([0., 0., 0., 0., 0.6, 2., 2., 2., 2.])
out = to_sympy(penalty, t9.copy())
assert np.allclose(np.asarray(out.value, float), penalty(t9.copy()))

t = sympy.IndexedBase("t")
I, J = axis_idx(0), axis_idx(1)
pinned = {t[0]: 0, t[1]: 0, t[2]: 0, t[3]: 0,          # left boundary
          t[6]: t[5], t[7]: t[5], t[8]: t[5]}           # right boundary
def code_entry(i, j):
    return sympy.cancel(out.formula.subs({I: i, J: j}).subs(pinned).doit())

Math(r"\Omega^{\mathrm{code}}_{1,2} = " + latex(code_entry(1, 2)))

<IPython.core.display.Math object>

Besides the formulas, the trace kept a record of every decision
the code made. The `if run != 0` checks became recorded conditions,
and reading them shows the code discovering the knot structure on its
own: the runs between repeated boundary knots are zero, all others are
not. A few of them:

In [3]:
pre = list(out.preconditions.args)
print(f"{len(pre)} conditions were recorded during the trace")
for g in pre[:4]:
    display(Math(latex(g)))

13 conditions were recorded during the trace


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Short and readable: the four left knots are all $0$ and the four
right knots are all $t_5$, so after filling those in, only two knots
remain, the interior one $t_4$ and the end $t_5$. No numbers anywhere,
this is the code's answer as a formula.

Now compute the same entry the other way, from the integral. We build
the basis functions from their textbook recursion, take two
derivatives, and integrate. The code is never consulted:

In [4]:
u = sympy.Symbol("u", positive=True)
knots = [0, 0, 0, 0, t[4], t[5], t[5], t[5], t[5]]
intervals = [(0, t[4]), (t[4], t[5])]

def basis(deg):
    if deg == 0:
        return [[sympy.Integer(1) if (knots[i] == lo and knots[i+1] == hi) else sympy.Integer(0)
                 for lo, hi in intervals] for i in range(len(knots) - 1)]
    prev = basis(deg - 1)
    out_ = []
    for i in range(len(knots) - 1 - deg):
        polys = []
        for seg in range(2):
            term = sympy.Integer(0)
            if knots[i + deg] != knots[i]:
                term += (u - knots[i]) / (knots[i + deg] - knots[i]) * prev[i][seg]
            if knots[i + deg + 1] != knots[i + 1]:
                term += (knots[i + deg + 1] - u) / (knots[i + deg + 1] - knots[i + 1]) * prev[i + 1][seg]
            polys.append(sympy.expand(term))
        out_.append(polys)
    return out_

B = basis(3)
def def_entry(i, j):
    val = sum(sympy.integrate(sympy.diff(B[i][seg], u, 2) * sympy.diff(B[j][seg], u, 2),
                              (u, lo, hi))
              for seg, (lo, hi) in enumerate(intervals))
    return sympy.cancel(sympy.together(val))

Math(r"\Omega^{\mathrm{def}}_{1,2} = " + latex(def_entry(1, 2)))

<IPython.core.display.Math object>

Same formula. That is already the story, but let's make it official:
subtract the two versions of all fifteen entries and ask sympy to
cancel. If everything cancels, the code and the integral agree for
every knot placement, not just some test values:

In [5]:
bad = 0
for i in range(5):
    for j in range(i, 5):
        if sympy.cancel(sympy.together(code_entry(i, j) - def_entry(i, j))) != 0:
            bad += 1
            print(f"entry ({i},{j}) differs")
print("Omega(code) == Omega(definition), identically:",
      "PROVED for all 0 < t4 < t5" if bad == 0 else f"{bad} entries differ")

Omega(code) == Omega(definition), identically: PROVED for all 0 < t4 < t5


## Straight lines are free

The report proves in prose that lines have zero penalty: constant and
linear splines have no curvature, so $\Omega$ must send them to zero.
The linear one is the spline whose coefficients are the Greville
points $g_j = (t_{j+1} + t_{j+2} + t_{j+3})/3$. The PR tests this at
tolerance 1e-12. Here it cancels exactly, as formulas:

In [6]:
Om = sympy.Matrix(5, 5, lambda i, j: code_entry(i, j))
ones = sympy.Matrix([1] * 5)
knots_pinned = [0, 0, 0, 0, t[4], t[5], t[5], t[5], t[5]]
g = sympy.Matrix([sympy.Rational(1, 3) * (knots_pinned[j + 1] + knots_pinned[j + 2]
                                          + knots_pinned[j + 3]) for j in range(5)])
display(Math(r"g = " + latex(g.T)))
z1 = [sympy.cancel(sympy.together(e)) for e in Om * ones]
zg = [sympy.cancel(sympy.together(e)) for e in Om * g]
print("Omega @ 1 =", z1, "-> exactly zero" if all(e == 0 for e in z1) else "")
print("Omega @ g =", zg, "-> exactly zero" if all(e == 0 for e in zg) else "")

<IPython.core.display.Math object>

Omega @ 1 = [0, 0, 0, 0, 0] -> exactly zero
Omega @ g = [0, 0, 0, 0, 0] -> exactly zero


**Proved:** the recipe equals the integral, as an identity, for
every knot vector with one interior knot; lines are exactly free.
**Not proved:** general n-knot vectors (the symbolic cost grows),
floating-point behavior (the identity lives in exact arithmetic; the
shipped function matched it to 1e-15 at every vector tried), and the
solver downstream of $\Omega$.

[The report](https://github.com/aadya940/scipy-bspline-testing/blob/main/B_Splines_with_arbitary_knots-gcv.pdf) took weeks. This takes a couple of minutes, and it runs
again whenever the code changes.

In [ ]:
# This same formula check now runs on every commit in tests/testing/test_penalty_matrix.py.
from skverify.testing import specifies

penalty_assumptions = [
    sympy.Eq(t[0], 0), sympy.Eq(t[1], 0), sympy.Eq(t[2], 0), sympy.Eq(t[3], 0),
    sympy.Eq(t[6], t[5]), sympy.Eq(t[7], t[5]), sympy.Eq(t[8], t[5]),
    t[4] > 0, t[5] > t[4],
]

notebook_knots = np.array([0., 0., 0., 0., 1., 3., 3., 3., 3.])

def penalty_entry_12(t):
    return penalty(t)[1, 2]

@specifies(-12 / (t[4]**2 * t[5]), assume=penalty_assumptions)
def notebook_penalty_entry_matches_the_paper():
    return penalty_entry_12, (notebook_knots.copy(),)

notebook_penalty_entry_matches_the_paper()
